# Обучение прототипа модели для прогнозирования средств клиентов

Конечно, для финального решения я напишу `train.py`, где будет реализовано распределенное обучение модели.
Но пока тут будет на `pandas`.

Планирую обучить $12$ моделей на каждый горизонт прогнозирования и посчитать метрики для каждой. Но пока обучу только на $1$ месяц вперед.

In [ ]:
import os
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error

import src.config as config

### Загружаем данные:

In [4]:
df = pd.read_parquet(config.SUPERVISED_PATH)
print(f"У нас {df.shape[0]} строк, {df.shape[1]} колонок")

У нас 2695629 строк, 38 колонок


### Фильтруем данные:

Тут мы выбираем только прогнозы на 1 месяц вперед, дропаем, где этот таргет null, дропаем строки, где `lag_12` null.

In [5]:
target_col = 'target_1m'

df_prepared = df.dropna(subset=[target_col, 'lag_12']).copy()

print(f"Строк осталось: {df_prepared.shape[0]}")

Строк осталось: 1929795


### Out-of-Time split:

Так как у нас именно временной ряд, то мы не можем использовать рандомный сплит, так как иначе есть шанс, что мы заглянем в будущее.

Нам по ТЗ представлены данные с 2022-01-31 по 2026-04-30, поэтому train - это вся история до февраля 2026 года включительно. Ну, а тест - это как раз март.

In [7]:
train_mask = df_prepared['report_date'] <= '2026-02-28'
test_mask = df_prepared['report_date'] == '2026-03-31'

train_data = df_prepared[train_mask]
test_data = df_prepared[test_mask]

print(f"Строк в обучающей выборке: {train_data.shape[0]}")
print(f"Строк в тестовой: {test_data.shape[0]}")

Строк в обучающей выборке: 1873293
Строк в тестовой: 56502


### Выделение признаков и таргета:

Тут мы выделяем именно фичи для модели, то есть убираем идентификаторы и прочий мусор. Остальные колонки таргеты также убираем.

In [9]:
drop_cols = ['client_id', 'report_date'] + [f'target_{i}m' for i in range(1, 13)]

X_train = train_data.drop(columns=drop_cols)
y_train = train_data[target_col]

X_test = test_data.drop(columns=drop_cols)
y_test = test_data[target_col]

cat_features = ['product', 'gender', 'segment', 'region']
for col in cat_features:
    X_train[col] = X_train[col].astype(str)
    X_test[col] = X_test[col].astype(str)

print(f"Итого у нас признаков: {X_train.shape[1]}")

Итого у нас признаков: 24


### Считаем метрики:

В ТЗ написано, что по-хорошему наша модель должна быть лучше бейзлайна, что "последнее известное фактическое значение по клиенту и продукту фиксируем на весь горизонт вперед".

Поэтому тут посчитаем **MAE** и **WAPE** для этого бейзлайна.

In [14]:
def wape(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred)) / np.sum(y_true)

baseline_preds = test_data['balance']

baseline_mae = mean_absolute_error(y_test, baseline_preds)
baseline_wape = wape(y_test, baseline_preds)

print(f"MAE: {baseline_mae:.2f} рублей")
print(f"WAPE: {baseline_wape * 100:.3f} процентов")

MAE: 14200.48 рублей
WAPE: 13.571 процентов


### Обучение модельки CatBoost:

In [15]:
model = CatBoostRegressor(
    iterations=600,
    learning_rate=0.05,
    depth=6,
    loss_function='MAE',
    random_seed=config.SEED,
    verbose=100
)

model.fit(
    X_train, y_train,
    cat_features=cat_features,
    eval_set=(X_test, y_test),
    plot=False
)

0:	learn: 97324.4883106	test: 95623.1888894	best: 95623.1888894 (0)	total: 959ms	remaining: 9m 34s
100:	learn: 18878.4684730	test: 18080.6703727	best: 18080.6703727 (100)	total: 1m 16s	remaining: 6m 16s
200:	learn: 14882.5211073	test: 14181.3397661	best: 14181.3397661 (200)	total: 2m 28s	remaining: 4m 55s
300:	learn: 14485.6166087	test: 13889.1400297	best: 13889.1400297 (300)	total: 3m 39s	remaining: 3m 38s
400:	learn: 13897.7488838	test: 13462.6944299	best: 13462.6944299 (400)	total: 4m 52s	remaining: 2m 25s
500:	learn: 13550.7421422	test: 13164.7409745	best: 13164.7409745 (500)	total: 6m 1s	remaining: 1m 11s
599:	learn: 13247.5317060	test: 12937.8017528	best: 12937.7664364 (598)	total: 7m 6s	remaining: 0us

bestTest = 12937.76644
bestIteration = 598

Shrink model to first 599 iterations.


CatBoostRegressor(depth=6, iterations=600, learning_rate=0.05, loss_function='MAE', random_seed=67, verbose=100)

### Оценка модели:

Так, ну по логам уже видно, что бейзлайн мы обогнали. Это успех!

In [16]:
model_preds = model.predict(X_test)
model_preds = np.clip(model_preds, 0, None)

model_mae = mean_absolute_error(y_test, model_preds)
model_wape = wape(y_test, model_preds)

print("Мини сравнение")
print(f"MAE: {model_mae:.2f} рублей (Бейзлайн: {baseline_mae:.2f})")
print(f"WAPE: {model_wape * 100:.3f} процентов (Бейзлайн: {baseline_wape * 100:.2f}%)")

Мини сравнение
MAE: 12937.27 рублей (Бейзлайн: 14200.48)
WAPE: 12.363 процентов (Бейзлайн: 13.57%)
